## Softmax Function - Multi-class Classification problem

Softmax is the generalization of the logistic regression function where the output of a sample could be one of n categories. Unlike logistic regression, here the samples belongs to more than 2 categories. However, if the classification is binary, the softmax reduces to logistic regression.

This function is used in both Softmax Regression and in Neural Networks when solving Multiclass Classification problems.

## Core idea behind Softmax Function

In both softmax regression and neural networks with Softmax outputs, N outputs are generated and one output is selected as the predicted category. In both cases a vector z is generated by a linear function which is applied to a softmax function. The softmax function converts z into a probability distribution. After applying softmax, each output will be between 0 and 1 and the outputs will add to 1, so that they can be interpreted as probabilities. The larger inputs will correspond to larger output probabilities.

In a Neural Network, if the output layer comprises a softmax activation, there could be more than just one neuron in the output layer. In such a case, each neuron would be responsible for predicting one of the class labels probability. For example, if the sample comprises one of the 10 possible categories, the output layer of a neural network would be constructed with 10 neurons, each forcomputing the corresponding class labels proabability.

The output of a softmax function is interpreted as the probability that the sample belongs to class label j, given features x, parameterized by weights w and b. This is repeated for probabilities of all possible class labels.

## Numpy Implementation

In [ ]:
# Importing the dependencies
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

In [2]:
def my_softmax(z):
    ez = np.exp(z)              #element-wise exponenial
    sm = ez/np.sum(ez)
    return(sm)

The softmax function differs from the sigmoid or ReLU activation function in a way that the traditional activation will take in one input (z) and produces one probabilistic output independent of the computations of other neurons in the layer, whereas in softmax activation, the computation of the neurons are more sort of dependent to each other. If any of the input z to the softmax function is altered, it will have an impact to the computations of other neurons in the layer, gradually leading to alteration in the output probability vector.

## Tensorflow Implementation

We will be implementing the softmax in two ways, cross-entropy loss in Tensorflow, the 'obvious' method and the 'preferred' method. The former is the most straightforward while the latter is more numerically stable.

Let's start by creating a dataset to train a multiclass classification model.

In [3]:
# creating dataset for example
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

## The Obvious organization

The model below is implemented with the softmax as an activation in the final Dense layer. The loss function is separately specified in the compile directive. The loss function is SparseCategoricalCrossentropy. In this model, the softmax takes place in the last layer. The loss function takes in the softmax output which is a vector of probabilities.

In [4]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < softmax activation here
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 1.0453 
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4071 
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1865 
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1090 
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0787 
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0642 
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0559 
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0503 
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0458 
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0423 


Because the softmax is integrated into the output layer, the output is a vector of probabilities.

In [5]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step  
[[2.08e-03 1.85e-03 9.84e-01 1.18e-02]
 [9.89e-01 1.00e-02 2.98e-04 3.54e-04]]
largest value 0.99999714 smallest value 3.57412e-11


## The Preferred Way

More stable and accurate results can be obtained if the softmax and loss are combined during training. This is enabled by the 'preferred' organization shown here.

In the preferred organization the final layer has a linear activation. For historical reasons, the outputs in this form are referred to as logits. The loss function has an additional argument: from_logits = True. This informs the loss function that the softmax operation should be included in the loss calculation. This allows for an optimized implementation.

In [6]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- Note
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7035 
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2775 
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1409 
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0909 
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0697 
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0586 
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0518 
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0469 
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0434 
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0407 


In the preferred model, the outputs are not probabilities, but can range from large negative numbers to large positive numbers. The output must be sent through a softmax when performing a prediction that expects a probability. Let's look at the preferred model outputs:

In [7]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step  
two example output vectors:
 [[-2.87 -1.46  2.81 -1.16]
 [ 5.1  -0.83 -5.06 -5.08]]
largest value 10.841249 smallest value -9.427114


The output predictions are not probabilities! If the desired output are probabilities, the output should be be processed by a softmax.

In [8]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[3.30e-03 1.34e-02 9.65e-01 1.82e-02]
 [9.97e-01 2.66e-03 3.84e-05 3.79e-05]]
largest value 0.9999956 smallest value 2.1536263e-08


To select the most likely category, the softmax is not required. We can simply find the index of the largest output using np.argmax().

In [9]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.87 -1.46  2.81 -1.16], category: 2
[ 5.1  -0.83 -5.06 -5.08], category: 0
[ 3.65 -0.15 -3.97 -4.11], category: 0
[-1.71  4.3  -1.98 -2.04], category: 1
[-2.23 -3.29  3.79 -4.93], category: 2


## SparseCategorialCrossentropy or CategoricalCrossEntropy

Tensorflow has two potential formats for target values and the selection of the loss defines which is expected.

SparseCategorialCrossentropy: expects the target to be an integer corresponding to the index. For example, if there are 10 potential target values, y would be between 0 and 9.

CategoricalCrossEntropy: Expects the target value of an example to be one-hot encoded where the value at the target index is 1 while the other N-1 entries are zero. An example with 10 potential target values, where the target is 2 would be [0,0,1,0,0,0,0,0,0,0].

## Final Outcome

Became more familiar with the softmax function and its use in softmax regression and in softmax activations in neural networks.

Learned the preferred model construction in Tensorflow:
No activation on the final layer (same as linear activation)
SparseCategoricalCrossentropy loss function
use from_logits=True

Recognized that unlike ReLU and Sigmoid, the softmax spans multiple outputs.